# ✈️ LLM Flight Assistant with Tool Calling

## Overview
This project demonstrates an end-to-end LLM application with:
- Multi-model support (Gemini + Ollama)
- Function/tool calling (dynamic ticket pricing)
- Conversational memory handling
- Interactive UI using Gradio

## Features
- Switch between cloud (Gemini) and local (Ollama) models
- Tool integration for real-world functionality
- Clean modular pipeline design

## Tech Stack
- Python
- OpenAI-compatible APIs (Gemini, Ollama)
- Gradio
- dotenv

## Learning Highlights
- Implemented tool/function calling in LLMs
- Built a modular chat pipeline
- Integrated multiple LLM providers in one app

In [1]:
import os
import json
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if google_api_key:
    print(f"API Key loaded: {google_api_key[:4]}***")
else:
    print("API Key not found")

API Key loaded: AIza***


In [2]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OLLAMA_URL = "http://localhost:11434/v1"

MODEL_GEMINI = "gemini-2.5-flash-lite"
MODEL_OLLAMA = "llama3.2:latest"

system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""


In [3]:
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
ollama = OpenAI(base_url=OLLAMA_URL, api_key="ollama")


In [4]:
ticket_prices = {
    "london": "$799",
    "paris": "$899",
    "tokyo": "$1400",
    "berlin": "$499"
}

In [5]:
def get_ticket_price(destination_city):
    print(f"Tool called for city: {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"

In [6]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "City user wants to travel to"
            }
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": price_function}]

In [7]:
def handle_tool_call(message):
    tool_call = message.tool_calls[0]

    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get("destination_city")

        result = get_ticket_price(city)

        return {
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id
        }


In [8]:
def chat(message, history, model_choice):

    history = [{"role": h["role"], "content": h["content"]} for h in history]

    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    # Select model
    if model_choice == "Gemini":
        client = gemini
        model = MODEL_GEMINI
    else:
        client = ollama
        model = MODEL_OLLAMA

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools if model_choice == "Ollama" else None
    )

    # Handle tool calls (only Ollama case)
    if response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message

        tool_response = handle_tool_call(message_obj)

        messages.append(message_obj)
        messages.append(tool_response)

        response = client.chat.completions.create(
            model=model,
            messages=messages
        )

    return response.choices[0].message.content


In [ ]:
def gradio_chat(message, history, model_choice):
    return chat(message, history, model_choice)

interface = gr.ChatInterface(
    fn=gradio_chat,
    additional_inputs=[
        gr.Dropdown(
            choices=["Gemini", "Ollama"],
            value="Ollama",
            label="Select Model"
        )
    ],
    title="✈️ FlightAI Assistant",
    description="Ask about flights, prices, and travel info"
)

interface.launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


Tool called for city: ["destination_city"]
Tool called for city: London
